# Topic 28 - Where RAG Fails

Companion notebook for `diagnose.py`. Given the corpus, retrieved, prompted and cited sets, classify the failure as unanswerable, retrieval miss, ranking miss, or generation miss - then attach causes (stale index, conflict, citation mismatch, over-abstention).

## Learning objectives

- run the topic's mechanics offline, cell by cell
- connect every demo to the pytest suite that pins it
- leave with numbers you can quote, not impressions


## 0. Imports and the labelled fixture

In [1]:
from dataclasses import replace

from diagnose import (
    BOUNDARIES, BOUNDARY_STAGES, LABELS, SAMPLE_CASES, SAMPLE_CHUNKS, STAGES,
    diagnose, extract_claims, failure_boundary, find_conflicts, guide,
    locate_stage, run_suite,
)

print('cases:', len(SAMPLE_CASES))
print('chunks:', len(SAMPLE_CHUNKS))
print('labels:', LABELS)

cases: 10
chunks: 8
labels: ('ok', 'retrieval_miss', 'ranking_miss', 'generation_miss', 'unanswerable')


## 1. Score the diagnostic suite

Ten cases covering every label and cause. `accuracy=1.00` means the diagnostic logic agrees with the human labels on the fixture.

In [2]:
report = run_suite(SAMPLE_CASES, chunks=SAMPLE_CHUNKS)
print(report.summary())
print()
for diagnosis in report.diagnoses:
    print(diagnosis.as_row())
assert report.accuracy == 1.0
assert report.mismatches == []

cases=10 scored=10 correct=10 accuracy=1.00
labels: generation_miss=4, ok=1, ranking_miss=1, retrieval_miss=2, unanswerable=2

ok                       ok               causes: -
retrieval-miss           retrieval_miss   causes: -
ranking-miss             ranking_miss     causes: prompt_pruned
generation-miss          generation_miss  causes: -
unanswerable             unanswerable     causes: -
stale-index              retrieval_miss   causes: stale_index
conflicting-chunks       generation_miss  causes: conflict
citation-mismatch        generation_miss  causes: citation_mismatch
over-abstention          generation_miss  causes: over_abstention
answered-unanswerable    unanswerable     causes: answered_unanswerable


## 2. The decision procedure

Corpus first, then retrieval, then prompt inclusion, then answer correctness. Once the gold chunk is outside the corpus, nothing downstream can be blamed.

In [3]:
print(f"{'in_corpus':<10}{'retrieved':<10}{'prompted':<10}{'correct':<9}label")
for retrieved in (False, True):
    for prompted in (False, True):
        for correct in (False, True):
            label = locate_stage(True, retrieved, prompted, correct)
            print(f"{str(True):<10}{str(retrieved):<10}{str(prompted):<10}{str(correct):<9}{label}")
print()
print('outside corpus ->', locate_stage(False, True, True, True))

in_corpus retrieved prompted  correct  label
True      False     False     False    retrieval_miss
True      False     False     True     retrieval_miss
True      False     True      False    retrieval_miss
True      False     True      True     retrieval_miss
True      True      False     False    ranking_miss
True      True      False     True     ranking_miss
True      True      True      False    generation_miss
True      True      True      True     ok

outside corpus -> unanswerable


## 3. Stale index: a cause, not a symptom

The label is `retrieval_miss` (the fresh chunk never came back). The cause is `stale_index` (45 days old). Fix the freshness pipeline, not the retriever.

In [4]:
stale = diagnose(next(c for c in SAMPLE_CASES if c.case_id == 'stale-index'), chunks=SAMPLE_CHUNKS)
print('label:', stale.label)
print('causes:', stale.causes)
print('notes:', stale.notes)
for fix in stale.fixes[:3]:
    print(' fix:', fix)
assert stale.label == 'retrieval_miss'
assert 'stale_index' in stale.causes

label: retrieval_miss
causes: ('stale_index',)
notes: gold=['returns#00']; in_corpus=True, retrieved=False, prompted=False; answer_correct=False, abstained=False; retrieved_n=2, prompted_n=2
 fix: check chunking: the answer may be split across chunks or buried in boilerplate
 fix: try hybrid retrieval (BM25 + vector) - exact names/IDs are where embeddings fail
 fix: verify ANN recall (ef_search / probes) or run exact search on the same query


## 4. Conflicting chunks

Two chunks state different values for the same fact. Both are relevant; the corpus is inconsistent. The model quoted the old value and the diagnosis is a generation miss with cause `conflict`.

In [5]:
conflict_case = next(c for c in SAMPLE_CASES if c.case_id == 'conflicting-chunks')
claims = extract_claims({cid: SAMPLE_CHUNKS[cid] for cid in ('policy#old', 'policy#new')})
for claim in claims:
    print('claim:', claim)
conflicts = find_conflicts(claims)
for conflict in conflicts:
    print('conflict:', conflict.subject, conflict.statements)
diagnosis = diagnose(conflict_case, chunks=SAMPLE_CHUNKS)
print('diagnosis:', diagnosis.label, diagnosis.causes)
assert conflicts and conflicts[0].subject == 'return window'
assert 'conflict' in diagnosis.causes

claim: Claim(chunk_id='policy#new', subject='return window', value=14, unit='day')
claim: Claim(chunk_id='policy#old', subject='return window', value=30, unit='day')
conflict: return window (('policy#new', '14 days'), ('policy#old', '30 days'))
diagnosis: generation_miss ('conflict',)


## 5. Unanswerable questions: one abstains, one hallucinates

Both cases have no gold chunk in the corpus. The correct behaviour is refusal; the second case answers fluently and is flagged `answered_unanswerable`.

In [6]:
for case_id in ('unanswerable', 'answered-unanswerable'):
    diagnosis = diagnose(next(c for c in SAMPLE_CASES if c.case_id == case_id), chunks=SAMPLE_CHUNKS)
    print(f'{case_id:<24} label={diagnosis.label:<12} causes={diagnosis.causes}')
hallucinated = diagnose(next(c for c in SAMPLE_CASES if c.case_id == 'answered-unanswerable'), chunks=SAMPLE_CHUNKS)
assert hallucinated.label == 'unanswerable'
assert 'answered_unanswerable' in hallucinated.causes

unanswerable             label=unanswerable causes=()
answered-unanswerable    label=unanswerable causes=('answered_unanswerable',)


## 6. Generation misses you can catch deterministically

A citation to a chunk that was never prompted; and a refusal even though the gold chunk reached the prompt (over-abstention).

In [7]:
for case_id in ('citation-mismatch', 'over-abstention'):
    diagnosis = diagnose(next(c for c in SAMPLE_CASES if c.case_id == case_id), chunks=SAMPLE_CHUNKS)
    print(f'{case_id:<22} {diagnosis.label:<16} {diagnosis.causes}')
assert 'citation_mismatch' in diagnose(next(c for c in SAMPLE_CASES if c.case_id == 'citation-mismatch')).causes
assert 'over_abstention' in diagnose(next(c for c in SAMPLE_CASES if c.case_id == 'over-abstention')).causes

citation-mismatch      generation_miss  ('citation_mismatch',)
over-abstention        generation_miss  ('over_abstention',)


## 7. Failure boundaries: what breaks / notice / change

One row per pipeline stage. If a row has no probe, the failure is a story, not something you can operate.

In [8]:
assert set(BOUNDARIES) == set(STAGES)
for stage in BOUNDARY_STAGES:
    boundary = failure_boundary(stage)
    print(f'{stage:<9} notice: {boundary.notice[:78]}')
    print(f'{"":<9} change: {boundary.change[:78]}')

ingest    notice: source-document count vs indexed-document count; per-document checksums; a zer
          change: assert counts and checksums, fail the job on empty sources, alert on ingest vo
chunk     notice: chunk-level recall on labelled questions; inspect the chunks around each gold 
          change: paragraph/semantic chunking, modest overlap, keep structured blocks intact, tu
embed     notice: vector norms, model id pinned per row, retrieval evals after every model chang
          change: pin model + task prefix, reject zero vectors, reindex on model change, evaluat
store     notice: freshness canary document, row counts per source, index build logs, age of new
          change: idempotent upserts keyed by chunk id, prune removed chunks, reindex triggers, 
query     notice: log the final query text and its norm; compare rewritten vs raw query retrieva
          change: guard query rewriting, fall back to the raw question, keep one embedding contr
retrieve  notice: recall@k vs 

## 8. Symptoms to first moves

Humans arrive with symptoms, not labels. The guide maps the common ones to a diagnosis and a first check.

In [9]:
for symptom in (
    'new documents',
    'two answers contradict each other',
    'the answer is plausible but cites nothing',
    'retrieval returns similar chunks but never the exact one',
):
    print(symptom)
    print('  ->', guide(symptom))
print()
assert 'stale_index' in guide('new documents')

new documents
  -> stale_index / ingest: run a freshness canary, compare source count to index count, and check the reindex trigger.
two answers contradict each other
  -> conflict (surfaces as generation_miss): two chunks state different values for the same fact; detect the conflict at assembly and prefer recency/authority or surface both.
the answer is plausible but cites nothing
  -> generation_miss / hallucination: require citations and run a groundedness check; abstain when no chunk supports the claim.
retrieval returns similar chunks but never the exact one
  -> retrieval_miss: inspect chunk boundaries and ANN recall; try hybrid search and a larger ef_search/probes.



## 9. The suite catches a mislabel

Change one expected label and the scorer reports the mismatch instead of quietly agreeing.

In [10]:
broken = replace(next(c for c in SAMPLE_CASES if c.case_id == 'retrieval-miss'), expected='generation_miss')
bad_report = run_suite([broken], chunks=SAMPLE_CHUNKS)
print(bad_report.summary())
assert bad_report.mismatches == ['retrieval-miss']
assert bad_report.accuracy == 0.0

cases=1 scored=1 correct=0 accuracy=0.00
labels: retrieval_miss=1
mismatches: retrieval-miss


## 10. Run the tests

In [11]:
import subprocess
import sys

result = subprocess.run([sys.executable, "-m", "pytest", "-q"], capture_output=True, text=True)
print(result.stdout[-1800:])
assert result.returncode == 0, "pytest failures!"
assert "72 passed" in result.stdout, "unexpected test count"
print("offline tests passed")


........................................................................ [100%]
72 passed in 0.06s

offline tests passed


## 11. What to notice

- four sets (corpus/retrieved/prompted/cited) are required to diagnose anything
- check corpus first, then retrieval, then prompt inclusion, then generation
- stale index and conflicts are **causes**, not labels
- unanswerable questions need a false-answer metric, not recall@k
- over-abstention is a generation failure too
- every stage needs a boundary row with a probe and a change
- the diagnostic logic itself is tested against labelled cases